# Chapter 21 – Worked examples

This notebook runs the examples of [Chapter 21](../index.md) live. The code lives in the modules next to it
(`flow_based.py`, `glsk_sensitivity.py`, `ieee14_flow_based.py`); the cells only call them, so a change in a
module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import flow_based
import glsk_sensitivity
import ieee14_flow_based as fb
import numpy as np

## Flow-based capacity calculation on the four-bus grid

Four buses with equal reactances, zone A = {0, 1}, zone B = {2, 3}, slack at bus 0. `flow_based_domain` returns,
per CNEC (one per branch, from-to direction), the zone-to-zone PTDF $\text{PTDF}_A - \text{PTDF}_B$, the
remaining available margin $\text{RAM} = F_\max - \text{FRM} - F_0$ and the A→B exchange at which the CNEC
reaches its margin.

In [ ]:
ptdf = flow_based.nodal_ptdf(flow_based.BRANCHES, 4)
ptdf_ab, ram, exchange = flow_based.flow_based_domain(
    ptdf, flow_based.GLSK, flow_based.P_REF, flow_based.F_MAX, flow_based.FRM
)
for branch, s, r, e in zip(flow_based.BRANCHES, ptdf_ab, ram, exchange, strict=True):
    print(f'branch {branch}: PTDF A->B {s:6.3f}   RAM {r:6.1f} MW   exchange limit {e:7.0f} MW')
k = int(exchange.argmin())
print(f'the market may add {exchange[k]:.0f} MW of A->B exchange before branch {flow_based.BRANCHES[k]} binds')

The nodal PTDF has a zero column for the slack: the zone-to-zone PTDF does not depend on that choice, because a
balanced exchange injects and withdraws the same power.

In [ ]:
print(np.round(ptdf, 3))

## GLSK sensitivity

The same grid with three GLSKs for zone A. The reference flows are physical and do not move; the zonal PTDFs,
$F_0$ and the margins do (Exercise 3).

In [ ]:
print(glsk_sensitivity.compare().round(3).to_string())

## The IEEE 14 grid as three zones

The Lab module builds the DC model from pypowsybl's IEEE 14, with zones A = {1–5}, B = {6, 11, 12, 13} and
C = {7, 8, 9, 10, 14}. The hand-built PTDF reproduces pypowsybl's DC flows to machine precision when the
transformer ratios are nominal (`dc_use_transformer_ratio=False`).

In [ ]:
grid = fb.load_ieee14()
h = fb.ptdf_matrix(grid)
print('net positions A, B, C:', np.round(fb.net_positions(grid), 1))
print('generation GLSK of zone A:', np.round(fb.glsk(grid, 'generation')[:5, 0], 3))